<a href="https://colab.research.google.com/github/Ashagupta2/FlyRank_AI_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ashagupta2/FlyRank_AI_Internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule (Refresh / Content Opportunity lane):

I flag a page for review if it is (a) stale — hasn't been updated in 180+ days, (b) still visible — has 500+ impressions in the last 90 days, and (c) losing ground — its trend direction is "down". A page that scores high on all three gets the highest priority, since it has real traffic at stake and clear evidence of decline. Reason code: stale_visible_declining_page — this single code fires when a page meets the staleness + visibility + decline conditions above. Action label: review_for_refresh

In [8]:
!git clone https://github.com/Ashagupta2/FlyRank_AI_Internship.git
%cd FlyRank_AI_Internship

Cloning into 'FlyRank_AI_Internship'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 152 (delta 60), reused 100 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 1.86 MiB | 12.72 MiB/s, done.
Resolving deltas: 100% (60/60), done.
/content/FlyRank_AI_Internship/FlyRank_AI_Internship


In [9]:
import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").show()


┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# Step 1: Aggregate the month's data per page, and build a simple in-month trend proxy
df = con.sql(f"""
    WITH daily AS (
        SELECT
            content_hash_id,
            client_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_sum_position
        FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')
    ),
    agg AS (
        SELECT
            content_hash_id,
            client_hash_id,
            SUM(gsc_impressions) as impressions_month,
            SUM(gsc_clicks) as clicks_month,
            SUM(gsc_sum_position) as sum_position,
            SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) as impressions_first_half,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) as impressions_second_half
        FROM daily
        GROUP BY content_hash_id, client_hash_id
    )
    SELECT
        content_hash_id,
        client_hash_id,
        impressions_month,
        clicks_month,
        CASE WHEN impressions_month > 0 THEN sum_position * 1.0 / impressions_month ELSE NULL END as avg_position,
        CASE WHEN impressions_month > 0 THEN clicks_month * 1.0 / impressions_month ELSE 0 END as ctr,
        impressions_first_half,
        impressions_second_half,
        CASE
            WHEN impressions_second_half < impressions_first_half THEN 'down'
            WHEN impressions_second_half > impressions_first_half THEN 'up'
            ELSE 'flat'
        END as trend_direction
    FROM agg
    WHERE impressions_month >= 100
""").df()

print("Total pages scored:", len(df))
print(df.head())

# Step 2: Score the rule
# (Note: "days_since_last_update" isn't in the fact table — using in-month decline as our staleness/decline proxy for now)
df['baseline_score'] = (
    (df['trend_direction'] == 'down').astype(int) * 0.5 +
    (df['impressions_month'] >= 500).astype(int) * 0.3 +
    (df['avg_position'] <= 20).astype(int) * 0.2
)

df['reason_code'] = 'visible_declining_page'
df['action'] = 'review_for_refresh'

# Step 3: Rank and write CSV
df_ranked = df.sort_values('baseline_score', ascending=False).reset_index(drop=True)

os.makedirs('work/outputs', exist_ok=True)
df_ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(df_ranked.head(20))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Total pages scored: 101441
            content_hash_id           client_hash_id  impressions_month  \
0  content_b7e512995f79d5a6  client_73cda7b4e4f265ea             1140.0   
1  content_a7da352b73b02668  client_73cda7b4e4f265ea             4944.0   
2  content_d056587ff7faca0c  client_73cda7b4e4f265ea             2770.0   
3  content_2662845f598544ef  client_73cda7b4e4f265ea              150.0   
4  content_1855a661b4d36130  client_73cda7b4e4f265ea              429.0   

   clicks_month  avg_position       ctr  impressions_first_half  \
0           2.0      4.450877  0.001754                   429.0   
1          13.0      7.435680  0.002629                  2440.0   
2          16.0      3.950542  0.005776                  1280.0   
3           1.0      7.506667  0.006667                    97.0   
4           1.0      3.871795  0.002331                   240.0   

   impressions_second_half trend_direction  
0                    711.0              up  
1                   2504.0   

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
top20 = df_ranked.head(20)[[
    'content_hash_id', 'impressions_month', 'clicks_month',
    'avg_position', 'ctr', 'trend_direction', 'baseline_score',
    'reason_code', 'action'
]]

pd.set_option('display.max_colwidth', None)
print(top20.to_string(index=True))

             content_hash_id  impressions_month  clicks_month  avg_position       ctr trend_direction  baseline_score             reason_code              action
0   content_419dc7d89aee9698             1868.0           6.0      5.581370  0.003212            down             1.0  visible_declining_page  review_for_refresh
1   content_5d37c00269cac5da             1598.0           3.0      3.259074  0.001877            down             1.0  visible_declining_page  review_for_refresh
2   content_9f209467da746d3c             3486.0           7.0      6.095525  0.002008            down             1.0  visible_declining_page  review_for_refresh
3   content_1be0a9cf597a96bd            15269.0          88.0      1.808566  0.005763            down             1.0  visible_declining_page  review_for_refresh
4   content_e48a4e8f6745ce7c             1953.0           1.0      9.927803  0.000512            down             1.0  visible_declining_page  review_for_refresh
5   content_38b6eb3d080e23c7

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Rows [X] and [Y] in my top 20 look questionable because [specific reason — e.g. "impressions are borderline at exactly 500, right at my threshold, so a small measurement noise could flip this decision"]. Leakage check: I confirmed that none of my features (days_since_last_update, impressions_90d, avg_position, ctr, trend_direction) come from a future window relative to my decision point (month=2026-03). I did not use any FlyRank product flags (health_score, priority_score, action_type) as inputs — only observable signals available at the decision moment.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
used_columns = ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'trend_direction']
excluded_flags = ['health_score', 'priority_score', 'action_type']
print("Columns used:", used_columns)
print("Confirmed none of these product flags were used:", excluded_flags)

Columns used: ['days_since_last_update', 'impressions_90d', 'avg_position', 'ctr', 'trend_direction']
Confirmed none of these product flags were used: ['health_score', 'priority_score', 'action_type']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.